In [ ]:
#Mounting a new folder from google colab onto drive
from google.colab import drive
drive.mount('/content/drive')


### Download ERA5 Meteorological Data

To download ERA5 data, we will use the Copernicus Climate Change Service (C3S) Climate Data Store (CDS) API. You will need to:

1.  **Register/Log in** to the [Climate Data Store](https://cds.climate.copernicus.eu/).
2.  **Generate an API key:** Go to your user profile page on the CDS website (https://cds.climate.copernicus.eu/user) and copy your `UID` and `API Key`.
3.  **Add to Colab Secrets:** In Colab, click the "🔑" (Secrets) icon in the left sidebar. Add two new secrets:
    *   `CDSAPI_UID` and paste your User ID.
    *   `CDSAPI_KEY` and paste your API Key.
    Make sure "Notebook access" is toggled on for both.

In [ ]:
print('Installing cdsapi...')
!pip install cdsapi -q
print('cdsapi installed successfully!')

import os
from google.colab import userdata

# Retrieve CDS API credentials from Colab secrets

CDSAPI_KEY = userdata.get('CDSAPI_KEY')

if CDSAPI_KEY:
    # Create the .cdsapirc file
    with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
        f.write(f'url: https://cds.climate.copernicus.eu/api')
        f.write(f'key: {CDSAPI_KEY}\n')
    print('CDS API credentials configured.')
else:
    print('Error: CDSAPI_KEY not found in Colab secrets. Please set it up as instructed.')

 """   "fire_name": "Sparks_Lake_fire_BC_2021",
    "date": datetime(2021, 8, 1),
    "start_hour_utc": 20,
    "end_hour_utc": 20,
    "fire_lat": 50.5,
    "fire_lon": -119.75,
    "crop_half_km": 100,
    "output_dir": "/content/drive/MyDrive/Pyrocb_data/ERA5_1",
"""

In [ ]:
import cdsapi
import os
from datetime import datetime, timedelta
import math

# Re-define CONFIG for clarity and self-containment, using values from cell eGh9oAt3p2ho
CONFIG = {


    "fire_name": "260",
    "date": datetime(2022, 6, 8),
    "start_hour_utc": 8,
    "end_hour_utc": 8,
    "fire_lat": 63.3,
    "fire_lon": -155.6,
    "crop_half_km": 100,
    "output_dir": "/content/drive/MyDrive/Pyrocb_data/ERA5_1",
}

# Explicitly pass URL and KEY to cdsapi.Client to bypass malformed .cdsapirc file
c = cdsapi.Client(url='https://cds.climate.copernicus.eu/api', key=CDSAPI_KEY)

# --- Extract parameters from CONFIG ---
fire_name = CONFIG['fire_name']
fire_date = CONFIG['date']
start_hour = CONFIG['start_hour_utc']
end_hour = CONFIG['end_hour_utc']
fire_lat = CONFIG['fire_lat']
fire_lon = CONFIG['fire_lon']
crop_half_km = CONFIG['crop_half_km']
base_output_dir = CONFIG['output_dir']

# --- Define output directory for ERA5 data ---
fire_dir = os.path.join(base_output_dir, fire_name)
era5_output_dir = os.path.join(fire_dir, "ERA5_data")
os.makedirs(era5_output_dir, exist_ok=True)
print(f"ERA5 data will be saved to: {era5_output_dir}")

# --- Calculate bounding box for ERA5 request ---
# Approximating 1 degree latitude = 111 km
# Approximating 1 degree longitude = 111 * cos(latitude) km
delta_lat = crop_half_km / 111.0
delta_lon = crop_half_km / (111.0 * math.cos(math.radians(fire_lat)))

# Bounding box: [north, west, south, east]
# CDS API expects latitudes in decreasing order (north to south)
area = [
    fire_lat + delta_lat,  # North
    fire_lon - delta_lon,  # West
    fire_lat - delta_lat,  # South
    fire_lon + delta_lon   # East
]
# Round to the nearest 0.25 as CDS API often expects specific coordinate granularity
area = [round(coord * 4) / 4 for coord in area]

# Generate list of hours (e.g., ['20:00', '21:00', '22:00', '23:00'])
hours_to_download = [f'{h:02d}:00' for h in range(start_hour, end_hour + 1)]

# --- Define ERA5 request parameters ---
request_params = {
    'product_type': ['reanalysis'],
    'data_format': ['grib'], # Changed to request GRIB format explicitly
    'variable': [
        '2m_temperature',
        'total_precipitation',
        'surface_pressure',
        '10m_u_component_of_wind',
        '10m_v_component_of_wind',
        'convective_inhibition',
        'geopotential',
        'surface_latent_heat_flux',
        'surface_sensible_heat_flux',
        'boundary_layer_height',
        'convective_available_potential_energy',
        '10m_wind_gust_since_previous_post_processing'

    ],
    'year': [str(fire_date.year)],
    'month': [str(fire_date.month).zfill(2)],
    'day': [str(fire_date.day).zfill(2)],
    'time': hours_to_download,
    'area': area,
}

# Adjust output filename for GRIB format
output_filename = os.path.join(era5_output_dir, f"ERA5_single_levels_{fire_name}_{fire_date.strftime('%Y%m%d')}_{start_hour:02d}-{end_hour:02d}UTC.grib")

print(f"\nRequesting ERA5 data for {fire_date.strftime('%Y-%m-%d')} from {start_hour:02d}:00 to {end_hour:02d}:00 UTC...")
print(f"Area: {area} (North, West, South, East)")
print(f"Variables: {request_params['variable']}")
print(f"Output file: {output_filename}")

try:
    c.retrieve(
        'reanalysis-era5-single-levels',
        request_params,
        output_filename
    )
    print(f"\nSuccessfully downloaded ERA5 data to: {output_filename}")
except Exception as e:
    print(f"\nError downloading ERA5 data: {e}")
    print("Please check your CDS API credentials and ensure the requested data is available for the specified parameters.")

In [ ]:
!pip install netcdf4 h5netcdf scipy -q
!pip install --upgrade cfgrib -q

import xarray as xr
import os
import netCDF4 # Import the netCDF4 library
import zipfile
import tempfile

# Load the downloaded ERA5 data

if os.path.exists(output_filename):
    file_size = os.path.getsize(output_filename)
    if file_size == 0:
        print(f"Error: The downloaded file '{output_filename}' is empty (0 bytes). This indicates a problem with the CDS API download, despite the success message. Please verify your request parameters and CDS API key.")
    else:
        print(f"\nAttempting to read the first few lines of '{output_filename}' as text for diagnostic purposes...")
        try:
            with open(output_filename, 'r', encoding='utf-8', errors='ignore') as f:
                head = [next(f) for _ in range(10)] # Read first 10 lines
            print("--- Start of file content (first 10 lines) ---")
            for line in head:
                print(line.strip())
            print("--- End of file content ---\n")
            print("This content might indicate if the file is an error message (e.g., HTML) instead of a NetCDF file, or hint at its actual format.")
        except Exception as e_read_text:
            print(f"Error reading file as text: {e_read_text}")
            print("Proceeding to attempt opening as GRIB/NetCDF...")

        # Initialize ds_era5 outside the try block
        ds_era5 = None

        if zipfile.is_zipfile(output_filename):
            print(f"\nDetected '{output_filename}' as a ZIP archive. Attempting to extract and open...")
            try:
                # Use TemporaryDirectory context manager to ensure cleanup
                with tempfile.TemporaryDirectory() as tmpdir:
                    print(f"Extracting to temporary directory: {tmpdir}")
                    with zipfile.ZipFile(output_filename, 'r') as zip_ref:
                        zip_ref.extractall(tmpdir)

                    extracted_files = [os.path.join(tmpdir, f) for f in os.listdir(tmpdir) if f.endswith(('.grib', '.grb', '.grib2', '.grb2', '.nc'))]

                    if extracted_files:
                        print(f"Found extracted files: {extracted_files}")
                        datasets = []
                        for grib_file_path in extracted_files:
                            print(f"Attempting to open extracted GRIB file: {grib_file_path}")
                            try:
                                # Open each extracted GRIB file *within* the temporary directory context
                                ds = xr.open_dataset(grib_file_path, engine='cfgrib')
                                datasets.append(ds)
                            except Exception as e_inner:
                                print(f"Error opening individual GRIB file {grib_file_path}: {e_inner}")

                        if datasets:
                            if len(datasets) > 1:
                                # Concatenate or merge if multiple datasets are found. Assuming they are meant to be combined.
                                ds_era5 = xr.merge(datasets)
                                print("\nSuccessfully opened and merged multiple GRIB files with xarray and 'cfgrib' engine.")
                            else:
                                ds_era5 = datasets[0]
                                print("\nSuccessfully opened single GRIB file with xarray and 'cfgrib' engine.")

                            print("\nERA5 Dataset Info:")
                            print(ds_era5)
                            print("\nFirst few records of ERA5 data:")
                            display(ds_era5.to_dataframe().head())
                        else:
                            print("No GRIB-like files could be successfully opened from the ZIP archive.")

                    else:
                        print("No GRIB-like files found inside the ZIP archive.")
            except Exception as e_zip_or_open:
                print(f"Error processing ZIP archive or opening extracted files: {e_zip_or_open}")
                print("Please check the integrity of the downloaded .grib.zip file and ensure 'cfgrib' is correctly installed and configured.")
        else: # Not a ZIP file, try to open directly as GRIB
            print(f"'{output_filename}' is not a ZIP archive. Attempting to open directly as GRIB...")
            try:
                ds_era5 = xr.open_dataset(output_filename, engine='cfgrib')
                print("\nSuccessfully opened file with xarray and 'cfgrib' engine.")
                print("\nERA5 Dataset Info:")
                print(ds_era5)
                print("\nFirst few records of ERA5 data:")
                display(ds_era5.to_dataframe())
            except Exception as e:
                print(f"Error opening dataset directly with xarray and cfgrib: {e}")
                print("The file might be corrupted or in an unexpected format. Please check the integrity of the downloaded .grib file and ensure 'cfgrib' is correctly installed and configured.")
else:
    print("ERA5 data file not found. Please ensure the download was successful.")

## Extracting Data for a Single Latitude and Longitude

Although you specified a bounding box for the download, the `ds_era5` dataset still contains the full grid of data within that box. To get data for a *single specific location* (your `fire_lat` and `fire_lon`), you need to `select` that point from the dataset.

We will use `ds_era5.sel()` with `method='nearest'` to find the closest grid point to your specified fire coordinates.

In [ ]:
if ds_era5 is not None:
    # Select the data for the specific fire latitude and longitude
    # 'method='nearest'' finds the closest grid point if the exact coordinate is not present
    fire_location_data = ds_era5.sel(latitude=fire_lat, longitude=fire_lon, method='nearest')

    print(f"\nData for fire location (lat: {fire_location_data.latitude.item()}, lon: {fire_location_data.longitude.item()}):")
    print(fire_location_data)

    print("\nData record at fire location (as DataFrame):")
    # Fix: Since we only have 1 hour and 1 location, the object is 0-dimensional.
    # We expand dimensions by 'time' to give to_dataframe() a valid index to work with.
    display(fire_location_data.expand_dims('time').to_dataframe())
else:
    print("ERA5 dataset (ds_era5) is not loaded, cannot extract data for fire location.")

### Batch Processing ERA5 Features
This section automates the extraction of ERA5 data for multiple fire events defined in your source CSV.

In [ ]:
import pandas as pd
import os
from datetime import datetime

# 0. Define Paths
INPUT_CSV = '/content/drive/MyDrive/Pyrocb_data/pyrocb_processed_results_combine1.csv'
OUTPUT_BASE_DIR = '/content/drive/MyDrive/Pyrocb_data/Era5_V/'
FINAL_CSV_PATH = '/content/drive/MyDrive/Pyrocb_data/era5_results.csv'

# 1. Load Data
df_events = pd.read_csv(INPUT_CSV)

# CONFIGURABLE: List the pyroCb_id(s) you want to process.
# Leave empty or set to None to process all.
TARGET_IDS = ['260', '258', '216', '189', '181', '180', '253', '179', '190', '202']

if TARGET_IDS:
    df_to_process = df_events[df_events['pyroCb_id'].astype(str).isin([str(i) for i in TARGET_IDS])].copy()
else:
    df_to_process = df_events.copy()

print(f"Found {len(df_to_process)} events to process.")
display(df_to_process.head())

In [ ]:
!pip install cdsapi cfgrib -q

import cdsapi
import xarray as xr
import math
import numpy as np
import pandas as pd
import os
import cfgrib
from google.colab import userdata

# Initialize CDS Client
CDSAPI_KEY = userdata.get('CDSAPI_KEY')
c = cdsapi.Client(url='https://cds.climate.copernicus.eu/api', key=CDSAPI_KEY)

def download_and_extract_era5(row):
    pyro_id = str(row['pyroCb_id'])
    # Parse timestamp
    ts = pd.to_datetime(row['timestamp'])
    date_str = ts.strftime('%Y-%m-%d')
    hour = ts.hour

    lat = round(row['pixel_latitude'], 2)
    lon = round(row['pixel_longitude'], 2)

    # Setup Directory
    event_dir = os.path.join(OUTPUT_BASE_DIR, pyro_id)
    os.makedirs(event_dir, exist_ok=True)
    grib_path = os.path.join(event_dir, f"{pyro_id}_{date_str}_{hour:02d}UTC.grib")

    # Calculate Area (100km buffer)
    crop_km = 100
    delta_lat = crop_km / 111.0
    delta_lon = crop_km / (111.0 * math.cos(math.radians(lat)))
    area = [round((lat + delta_lat)*4)/4, round((lon - delta_lon)*4)/4,
            round((lat - delta_lat)*4)/4, round((lon + delta_lon)*4)/4]

    # Download if not exists
    if not os.path.exists(grib_path):
        print(f"\n[LOG] Downloading data for ID {pyro_id} at {date_str} {hour:02d}:00")
        request = {
            'product_type': 'reanalysis',
            'data_format': 'grib',
            'variable': [
                'high_vegetation_cover',
                'low_vegetation_cover',
                'type_of_high_vegetation',
                'type_of_low_vegetation'
            ],
            'year': str(ts.year), 'month': f"{ts.month:02d}", 'day': f"{ts.day:02d}",
            'time': f"{hour:02d}:00", 'area': area,
        }
        c.retrieve('reanalysis-era5-single-levels', request, grib_path)

    # Extract Features using the robust merge logic
    print(f"[LOG] Extracting features for ID {pyro_id} at {date_str} {hour:02d}:00")

    # 1. Load Analysis and Forecast groups separately
    datasets = []
    for dtype in ['an', 'fc']:
        try:
            ds = xr.open_dataset(grib_path, engine='cfgrib', backend_kwargs={'filter_by_keys': {'dataType': dtype}})
            datasets.append(ds)
        except Exception:
            continue

    # 2. Merge using override to solve the 'time' coordinate conflict
    if len(datasets) > 1:
        ds_merged = xr.merge(datasets, compat='override')
    elif len(datasets) == 1:
        ds_merged = datasets[0]
    else:
        raise ValueError("Could not load any data from GRIB file.")

    # 3. Select nearest point
    point_data = ds_merged.sel(latitude=lat, longitude=lon, method='nearest')

    # 4. Use valid_time for index
    df_point = point_data.expand_dims('valid_time').to_dataframe().reset_index()

    # Add Metadata
    df_point['pyroCb_id'] = pyro_id
    return df_point

# Processing Loop
all_results = []
for index, row in df_to_process.iterrows():
    try:
        res = download_and_extract_era5(row)
        all_results.append(res)
    except Exception as e:
        print(f"[ERROR] Failed to process {row['pyroCb_id']} at {row['timestamp']}: {e}")

if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    final_df.to_csv(FINAL_CSV_PATH, index=False)
    print(f"\n[LOG] Success! Results saved to {FINAL_CSV_PATH}")
    display(final_df.head())
else:
    print("[LOG] No data was processed.")

In [ ]:
print('Displaying final_df with selected columns:')

desired_columns = ['valid_time', 'number', 'time', 'step', 'surface', 'latitude', 'longitude', 'cvh', 'cvl', 'tvh', 'tvl', 'pyroCb_id']

# Filter for columns that actually exist in the DataFrame
existing_columns = [col for col in desired_columns if col in final_df.columns]

final_df_selected = final_df[existing_columns]
final_df_selected.to_csv(FINAL_CSV_PATH, index=False)
print(f"\n[LOG] Success! Selected results saved to {FINAL_CSV_PATH}")
display(final_df_selected.head())

In [ ]:
FEATURES_CSV_PATH = '/content/merged_pyrocb_era5_features[1].csv'
df_existing_features = pd.read_csv(FEATURES_CSV_PATH)

# First, create the desired 'time' column from 'timestamp' with a temporary name
if 'timestamp' in df_existing_features.columns:
    df_existing_features['temp_time_for_merge'] = pd.to_datetime(df_existing_features['timestamp'])
else:
    raise ValueError("DataFrame 'df_existing_features' does not contain a 'timestamp' column.")

# Now, identify and drop all columns named 'time' and the original 'timestamp' column
cols_to_drop = [col for col in df_existing_features.columns if col == 'time' or col == 'timestamp']
df_existing_features = df_existing_features.drop(columns=cols_to_drop, errors='ignore')

# Finally, rename the temporary column to 'time'
df_existing_features = df_existing_features.rename(columns={'temp_time_for_merge': 'time'})

print("Head of existing features DataFrame (after cleaning 'time' column):")
display(df_existing_features.head())

print("Columns and dtypes of existing features DataFrame (after cleaning 'time' column):\n")
print(df_existing_features.info())

Head of existing features DataFrame (after cleaning 'time' column):


,pyroCb_id,pixel_latitude,pixel_longitude,dist_km,fire_proxy,cloud_height_proxy,raw_fire_bt,raw_cloud_bt,simulated_green,valid_time,...,slhf,sshf,fg10,wind_speed10,wind_dir_deg,cin_filled,injection_potential,PII,capped_flag,time
0,179,33.287442,-108.588483,1.75952,-90.97516,2.121407,0.082010,0.450477,91.425640,2021-05-27 06:00:00,...,-9385.0,30882.0,4.455146,2.046433,118.403705,0.0,-0.214816,-0.019456,False,2021-05-27 06:00:00
1,179,33.287442,-108.588483,1.75952,-97.58934,1.584457,0.034433,0.517745,98.107086,2021-05-27 12:00:00,...,-11440.0,53303.0,3.693646,1.874933,199.876023,0.0,-0.405668,-0.215182,False,2021-05-27 12:00:00
2,179,33.287442,-108.588483,1.75952,-98.93545,0.554916,32.590084,0.606913,99.542360,2021-05-27 18:00:00,...,-257449.0,-1756259.0,11.259994,3.837899,49.726855,0.0,-0.183975,-0.223115,False,2021-05-27 18:00:00
3,179,33.287442,-108.588483,1.75952,-99.40178,0.950851,24.936518,0.536517,99.938290,2021-05-28 00:00:00,...,-181576.0,-773753.0,13.473206,4.348494,56.359226,0.0,-0.199941,-0.204530,False,2021-05-28 00:00:00
4,179,33.287442,-108.588483,1.75952,-99.15588,2.111855,0.002714,0.534953,99.690834,2021-05-28 06:00:00,...,-7218.0,30642.0,4.098062,2.365244,149.384452,0.0,-0.449490,-0.247899,False,2021-05-28 06:00:00


Columns and dtypes of existing features DataFrame (after cleaning 'time' column):

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225 entries, 0 to 224
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   pyroCb_id            225 non-null    int64         
 1   pixel_latitude       225 non-null    float64       
 2   pixel_longitude      225 non-null    float64       
 3   dist_km              225 non-null    float64       
 4   fire_proxy           224 non-null    float64       
 5   cloud_height_proxy   224 non-null    float64       
 6   raw_fire_bt          224 non-null    float64       
 7   raw_cloud_bt         224 non-null    float64       
 8   simulated_green      224 non-null    float64       
 9   valid_time           225 non-null    object        
 10  number               225 non-null    int64         
 11  step                 225 non-null    object        
 12  surface  

In [ ]:
print("Attempting to merge DataFrames...")

# Ensure pyroCb_id is of consistent type across both dataframes for merging
final_df_selected['pyroCb_id'] = final_df_selected['pyroCb_id'].astype(str).copy()
df_existing_features['pyroCb_id'] = df_existing_features['pyroCb_id'].astype(str).copy()

# Ensure 'time' column in final_df_selected is a datetime object for accurate merging
final_df_selected['time'] = pd.to_datetime(final_df_selected['time']).copy()
# df_existing_features['time'] is already ensured to be datetime in the previous cell.

# Merge the two dataframes on 'pyroCb_id' and 'time'
# Using an outer merge to keep all records from both dataframes and see where matches occur
merged_df = pd.merge(final_df_selected, df_existing_features, on=['pyroCb_id', 'time'], how='outer', suffixes=('_vegetation', '_existing'))

print("Merge successful. Head of the merged DataFrame:")
display(merged_df.head())

print("Info of the merged DataFrame:")
print(merged_df.info())

Attempting to merge DataFrames...
Merge successful. Head of the merged DataFrame:


/tmp/ipykernel_1894/3602361487.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df_selected['pyroCb_id'] = final_df_selected['pyroCb_id'].astype(str).copy()
/tmp/ipykernel_1894/3602361487.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df_selected['time'] = pd.to_datetime(final_df_selected['time']).copy()


,valid_time_vegetation,number_vegetation,time,step_vegetation,surface_vegetation,latitude_vegetation,longitude_vegetation,cvh,cvl,tvh,...,cin,slhf,sshf,fg10,wind_speed10,wind_dir_deg,cin_filled,injection_potential,PII,capped_flag
0,2021-05-27 06:00:00,0,2021-05-27 06:00:00,0 days,0.0,33.25,-108.5,0.888824,0.093579,3.0,...,NaN,-9385.0,30882.0,4.455146,2.046433,118.403705,0.0,-0.214816,-0.019456,False
1,2021-05-27 12:00:00,0,2021-05-27 12:00:00,0 days,0.0,33.25,-108.5,0.888824,0.093579,3.0,...,NaN,-11440.0,53303.0,3.693646,1.874933,199.876023,0.0,-0.405668,-0.215182,False
2,2021-05-27 18:00:00,0,2021-05-27 18:00:00,0 days,0.0,33.25,-108.5,0.888824,0.093579,3.0,...,NaN,-257449.0,-1756259.0,11.259994,3.837899,49.726855,0.0,-0.183975,-0.223115,False
3,2021-05-28 00:00:00,0,2021-05-28 00:00:00,0 days,0.0,33.25,-108.5,0.888824,0.093579,3.0,...,NaN,-181576.0,-773753.0,13.473206,4.348494,56.359226,0.0,-0.199941,-0.204530,False
4,2021-05-28 06:00:00,0,2021-05-28 06:00:00,0 days,0.0,33.25,-108.5,0.888824,0.093579,3.0,...,NaN,-7218.0,30642.0,4.098062,2.365244,149.384452,0.0,-0.449490,-0.247899,False


Info of the merged DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227 entries, 0 to 226
Data columns (total 44 columns):
 #   Column                 Non-Null Count  Dtype          
---  ------                 --------------  -----          
 0   valid_time_vegetation  227 non-null    datetime64[ns] 
 1   number_vegetation      227 non-null    int64          
 2   time                   227 non-null    datetime64[ns] 
 3   step_vegetation        227 non-null    timedelta64[ns]
 4   surface_vegetation     227 non-null    float64        
 5   latitude_vegetation    227 non-null    float64        
 6   longitude_vegetation   227 non-null    float64        
 7   cvh                    227 non-null    float32        
 8   cvl                    227 non-null    float32        
 9   tvh                    227 non-null    float32        
 10  tvl                    227 non-null    float32        
 11  pyroCb_id              227 non-null    object         
 12  pixel_latitude      

In [ ]:
OUTPUT_MERGED_CSV_PATH = os.path.join(OUTPUT_BASE_DIR, 'complete_era5_veg.csv')
merged_df.to_csv(OUTPUT_MERGED_CSV_PATH, index=False)
print(f"Successfully saved merged_df to: {OUTPUT_MERGED_CSV_PATH}")

Successfully saved merged_df to: /content/drive/MyDrive/Pyrocb_data/Era5_V/complete_era5_veg.csv
